## **2. Feature Engineering**

### Token Level Features
&emsp;a) Imports & Load Data

In [3]:
import pandas as pd
import numpy as np
from transformers import MarianTokenizer, MarianMTModel

In [4]:
df = pd.read_csv('data/fr_en_cleaned_train.csv')
df.head(5)

,user_id,sent_id,token,POS,Morpho-Syntactic_Features,Dependency-Relation,Dependancy-Head,time,p_recall,sentence
0,YjS/mQOx,8XTyQUAl01,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,det,2,14.0,0,Le garçon
1,YjS/mQOx,8XTyQUAl01,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,ROOT,0,14.0,0,Le garçon
2,YjS/mQOx,8XTyQUAl02,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,nsubj,4,14.0,0,Je suis une femme
3,YjS/mQOx,8XTyQUAl02,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,cop,4,14.0,0,Je suis une femme
4,YjS/mQOx,8XTyQUAl02,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,det,4,14.0,0,Je suis une femme


In [5]:
tokens = df[['token', 'POS', 'Morpho-Syntactic_Features', 'sentence']]
tokens = tokens.drop_duplicates()
tokens.head(10)
print(tokens['token'].is_unique)

duplicate_rows = tokens[tokens.duplicated(subset=['token'], keep=False)]
print(duplicate_rows[duplicate_rows['token'] == 'suis'])
# i am preserving dup tokens because they have different POS and morpho-syntactic features which could be important for the model to learn. For example "suis" can be a verb (first person singular of "être") or a noun (plural of "sui" which is a type of plant). Removing duplicates would result in loss of information that could be relevant for the model to learn the correct associations between tokens, their POS tags, and their morpho-syntactic features.

False
       token   POS                          Morpho-Syntactic_Features  \
3       suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
9       suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
13      suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
16      suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
158     suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
...      ...   ...                                                ...   
706231  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
706249  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
738926  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
740263  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   
764653  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...   

                                 sentence  
3                       Je suis une femme  
9                       Je su

&emsp;b) Supplment Token Level Data
 * Specifically, I am using the Lexique data base to add information about syllable count, general word frequency to give allusion to how complex or uncommon the word is

In [6]:
lexique = pd.read_csv('data/OpenLexicon.tsv', sep='\t')
lexique = lexique.rename(columns={'ortho': 'token', 'Lexique4__Lemme' : 'lemma','Lexique4__SyllNb' : 'syllable_count'})
lexique.head(5)

,token,lemma,Lexique4__Cgram,Lexique4__CgramOrtho,Lexique4__FreqOrtho,syllable_count
0,a,a,NOM,"NOM,AUX,VER",11684.443,1
1,a,avoir,VER,"NOM,AUX,VER",11684.443,1
2,a,avoir,AUX,"NOM,AUX,VER",11684.443,1
3,a capella,a capella,ADV,ADV,0.244,4
4,a cappella,a cappella,ADV,ADV,0.241,4


 * The frequency column is how many times a word occurs out of a million instances. This is heavily skewed with a range of 11000+ to smaller that 0.3. For this reason I have decided to normalize the with column with a log transformation.

In [7]:
lexique['ortho_freq'] = np.log1p(lexique['Lexique4__FreqOrtho'])
tokens = pd.merge(
    tokens.assign(token_lower=tokens['token'].str.lower()),
    lexique[['token', 'syllable_count', 'ortho_freq']].rename(columns={'token': 'token_lower'}),
    on='token_lower',
    how='left'
).drop(columns='token_lower').drop_duplicates()

tokens.head(10)



,token,POS,Morpho-Syntactic_Features,sentence,syllable_count,ortho_freq
0,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,Le garçon,1.0,9.713540
3,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,Le garçon,2.0,5.244537
4,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis une femme,1.0,10.163428
6,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis une femme,1.0,8.330842
10,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,Je suis une femme,1.0,9.039167
14,femme,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,Je suis une femme,1.0,6.585643
15,La,DET,Definite=Def|Gender=Fem|Number=Sing|fPOS=DET++,La fille,1.0,9.613319
19,fille,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,La fille,1.0,6.447431
20,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,Je suis un garçon,1.0,10.163428
22,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,Je suis un garçon,1.0,8.330842


 * There is a lot of interesting linguistic information that may have an effect on learner retention if we can **identify the lemma of the token**. Second language aquisition research suggests that learners of a language have a tendency to retain words that are congugates or closer relatives to one in their native tongue. This makes sense and I want to take advantage of this by engineering a feature column that describes the **edit distance** from the french word to english. As the lessons used in this data set are from an English based course I continue under the assumption users are more familiar with English and can use this knowledge to make inferences on French words. Although, I may consider reducing down to countries where this is particularly plausible (US, CA, etc.)

In [ ]:
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-fr-en")

sentences = tokens['sentence'].unique().tolist()
fr_en_sentences = {}
for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt") # tokenize and convert to PyTorch tensors
    outputs = model.generate(**inputs) # "**" syntax unpacks inputs dict and output is tensor of english translation token ids
    translation = tokenizer.decode(outputs, skip_special_tokens=True)[0] # decode token ids to get english translation string
    fr_en_sentences[sentence] = translation

Loading weights: 100%|██████████| 256/256 [00:00<00:00, 17171.90it/s]


In [ ]:
fr_en_sentences

{'Le garçon': ['The boy'],
 'Je suis une femme': ["I'm a woman."],
 'La fille': ['The girl'],
 'Je suis un garçon': ["I'm a boy."],
 'Je suis riche': ["I'm rich."],
 'Je suis rouge': ["I'm red."],
 "L' homme": ['Man'],
 "L' homme est riche": ['The man is rich'],
 'Une pomme': ['Apple'],
 'Tu es un garçon': ["You're a boy."],
 'Je mange': ["I'm eating."],
 'Il est riche': ["He's rich."],
 'Elle est calme': ["She's calm."],
 'Il a un enfant': ['He has a child.'],
 'Il mange et je mange': ['He eats and I eat'],
 'La robe est rouge': ['The dress is red'],
 'Le chat est noir': ['The cat is black'],
 'Les hommes': ['Men'],
 'Nous sommes une femme et un homme': ['We are a woman and a man'],
 'Les hommes sont riches': ['Men are rich'],
 'Les hommes sont calmes': ['The men are calm.'],
 "J' écris": ["I'm writing."],
 "J' aime un garçon": ['I love a boy'],
 'Nous mangeons': ['We eat'],
 'Vous êtes calme': ["You're calm."],
 'Nous avons une pomme': ['We have an apple.'],
 'Le journal': ['The news